In [1]:
!pip install ydf -U

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [8]:
import pandas as pd
import numpy as np
import ydf
from ydf import CartLearner, Task, Semantic
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

# Chargement des fichiers
dataset_olt = '/content/drive/MyDrive/nexialog/data_with_anomalies_OLT_with_jump_and_DB_SCAN.parquet'
dataset_peag = "/content/drive/MyDrive/nexialog/data_with_anomalies_PEAG_with_jump_and_DB_SCAN.parquet"
df_olt = pd.read_parquet(dataset_olt)
df_peag = pd.read_parquet(dataset_peag)

# Cibles
target_cols = [
    'is_jump_avg_dns_time', 'is_jump_avg_latence_scoring', 'is_jump_avg_score_scoring',
    'is_anomaly_dns', 'is_anomaly_latence', 'is_anomaly_scoring'
]

# Colonnes à exclure dans tous les cas
always_exclude = [
    'missing_avg_dns_time', 'missing_std_dns_time',
    'missing_avg_latence_scoring', 'missing_std_latence_scoring',
    'missing_avg_score_scoring', 'missing_std_score_scoring',
    'code_departement', 'date_hour', 'peag_nro',
    'boucle', 'olt_name'
]

# Categorical définies
categorical_cols = [
    'olt_model', 'dsp', 'pop_dns',
    'moment_journee', 'depts_egaux', 'meme_region',
    'type_boucle', 'type_olt_model', 'week_end'
]

# Colonnes gardées par défaut
default_features = [
    'olt_model', 'dsp', 'pop_dns', 'nb_client_total',
    'moment_journee', 'depts_egaux', 'meme_region',
    'type_boucle', 'type_olt_model',
    'encodage_heure_sin', 'encodage_heure_cos',
    'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos',
    'week_end',
    'nb_test_dns', 'avg_dns_time', 'std_dns_time',
    'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring',
    'avg_score_scoring', 'std_score_scoring'
]

# Cibles spécifiques à exclure par target
target_specific_exclude = {
    'is_jump_avg_dns_time': ['nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring', 'avg_score_scoring', 'std_score_scoring'],
    'is_jump_avg_latence_scoring': ['nb_test_dns', 'avg_dns_time', 'std_dns_time', 'avg_score_scoring', 'std_score_scoring'],
    'is_jump_avg_score_scoring': ['nb_test_dns', 'avg_dns_time', 'std_dns_time', 'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring'],
    'is_anomaly_dns': ['nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring', 'avg_score_scoring', 'std_score_scoring'],
    'is_anomaly_latence': ['nb_test_dns', 'avg_dns_time', 'std_dns_time', 'avg_score_scoring', 'std_score_scoring'],
    'is_anomaly_scoring': ['nb_test_dns', 'avg_dns_time', 'std_dns_time', 'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring']
}


def get_features_and_semantics(df, target_col):
    all_excluded = set(always_exclude + target_specific_exclude.get(target_col, []) + [col for col in target_cols if col != target_col])
    selected_cols = [col for col in default_features if col not in all_excluded and col in df.columns]

    df_model = df[selected_cols + [target_col]].dropna(subset=[target_col]).copy()
    df_model["target"] = df_model[target_col].astype(int)
    df_model.drop(columns=[target_col], inplace=True)

    print(f"\n--- Target : {target_col} ---")
    print("Colonnes gardées :", list(df_model.columns))

    feature_semantics = []
    for col in df_model.columns:
        if col == "target":
            continue
        elif col in categorical_cols:
            feature_semantics.append((col, Semantic.CATEGORICAL))
        else:
            feature_semantics.append((col, Semantic.NUMERICAL))

    return df_model, feature_semantics


def build_and_evaluate_cart(df_model, feature_semantics, label, title):
    print(f"\n=== Entraînement modèle : {title} ===")
    train_df, test_df = train_test_split(df_model, test_size=0.2, stratify=df_model[label], random_state=42)

    learner = CartLearner(
        label=label,
        task=Task.CLASSIFICATION,
        features=feature_semantics,
        include_all_columns=True,
        max_depth=5
    )

    model = learner.train(train_df)
    model.describe()
    evaluation = model.evaluate(test_df, label=label)
    print(evaluation)
    display(model.plot_tree())
    print("-" * 80)


print("== Entraînement modèles OLT ==")
for target_col in target_cols:
    df_model, semantics = get_features_and_semantics(df_olt, target_col)
    build_and_evaluate_cart(df_model, semantics, "target", f"OLT - {target_col}")


print("== Entraînement modèles PEAG ==")
for target_col in target_cols:
    df_model, semantics = get_features_and_semantics(df_peag, target_col)
    build_and_evaluate_cart(df_model, semantics, "target", f"PEAG - {target_col}")



== Entraînement modèles OLT ==

--- Target : is_jump_avg_dns_time ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'nb_test_dns', 'avg_dns_time', 'std_dns_time', 'target']

=== Entraînement modèle : OLT - is_jump_avg_dns_time ===
Train model on 2973073 examples
Model trained in 0:00:07.200426
accuracy: 0.996747
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 737748 |    941 |
    +--------+--------+--------+
    |      1 |   1477 |   3103 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.933849
    PR AUC: 0.754211
    Num thresholds: 6
loss: 0.0122193
num examples: 743269
num examples (weighted): 743269



--------------------------------------------------------------------------------

--- Target : is_jump_avg_latence_scoring ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring', 'target']

=== Entraînement modèle : OLT - is_jump_avg_latence_scoring ===
Train model on 2973073 examples
Model trained in 0:00:05.810947
accuracy: 0.994718
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 737572 |    785 |
    +--------+--------+--------+
    |      1 |   3141 |   1771 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.936592
    PR AUC: 0.578907
    Num threshold

--------------------------------------------------------------------------------

--- Target : is_jump_avg_score_scoring ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'avg_score_scoring', 'std_score_scoring', 'target']

=== Entraînement modèle : OLT - is_jump_avg_score_scoring ===
Train model on 2973073 examples
Model trained in 0:00:05.743264
accuracy: 0.994378
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 739090 |      0 |
    +--------+--------+--------+
    |      1 |   4179 |      0 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.5
    PR AUC: 0.502811
    Num thresholds: 3
loss: 0.0347365
num example

--------------------------------------------------------------------------------

--- Target : is_anomaly_dns ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'nb_test_dns', 'avg_dns_time', 'std_dns_time', 'target']

=== Entraînement modèle : OLT - is_anomaly_dns ===
Train model on 2973073 examples
Model trained in 0:00:07.148140
accuracy: 0.996455
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 737708 |   1180 |
    +--------+--------+--------+
    |      1 |   1455 |   2926 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.868155
    PR AUC: 0.698591
    Num thresholds: 9
loss: 0.0156053
num examples: 743269
nu

--------------------------------------------------------------------------------

--- Target : is_anomaly_latence ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring', 'target']

=== Entraînement modèle : OLT - is_anomaly_latence ===
Train model on 2973073 examples
Model trained in 0:00:05.779774
accuracy: 0.99654
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 738646 |   1070 |
    +--------+--------+--------+
    |      1 |   1502 |   2051 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.906494
    PR AUC: 0.547904
    Num thresholds: 7
loss: 0.012981

...

--------------------------------------------------------------------------------

--- Target : is_jump_avg_score_scoring ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'avg_score_scoring', 'std_score_scoring', 'target']

=== Entraînement modèle : PEAG - is_jump_avg_score_scoring ===
Train model on 2973073 examples
Model trained in 0:00:05.715265
accuracy: 0.994378
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 739090 |      0 |
    +--------+--------+--------+
    |      1 |   4179 |      0 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.5
    PR AUC: 0.502811
    Num thresholds: 3
loss: 0.0347365
num exampl

--------------------------------------------------------------------------------

--- Target : is_anomaly_dns ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'nb_test_dns', 'avg_dns_time', 'std_dns_time', 'target']

=== Entraînement modèle : PEAG - is_anomaly_dns ===
Train model on 2973073 examples
Model trained in 0:00:07.093970
accuracy: 0.996201
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 737477 |   1411 |
    +--------+--------+--------+
    |      1 |   1413 |   2968 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.870046
    PR AUC: 0.693363
    Num thresholds: 9
loss: 0.0156118
num examples: 743269
n

--------------------------------------------------------------------------------

--- Target : is_anomaly_latence ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring', 'target']

=== Entraînement modèle : PEAG - is_anomaly_latence ===
Train model on 2973073 examples
Model trained in 0:00:05.761878
accuracy: 0.996283
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 739413 |    303 |
    +--------+--------+--------+
    |      1 |   2460 |   1093 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.918148
    PR AUC: 0.594925
    Num thresholds: 6
loss: 0.0128

--------------------------------------------------------------------------------

--- Target : is_anomaly_scoring ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'avg_score_scoring', 'std_score_scoring', 'target']

=== Entraînement modèle : PEAG - is_anomaly_scoring ===
Train model on 2973073 examples
Model trained in 0:00:05.706264
accuracy: 0.998624
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 742246 |      0 |
    +--------+--------+--------+
    |      1 |   1023 |      0 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.5
    PR AUC: 0.500688
    Num thresholds: 3
loss: 0.0104433
num examples: 743269
num

--------------------------------------------------------------------------------


Entrainement et prédictions des arbres avec colonnes uniquement contextuelles/ catégorielles

In [9]:
# === VERSION TEST-FREE : Entraînement sans aucune variable de test ===

# Cibles à prédire
target_cols = [
    'is_jump_avg_dns_time', 'is_jump_avg_latence_scoring', 'is_jump_avg_score_scoring',
    'is_anomaly_dns', 'is_anomaly_latence', 'is_anomaly_scoring'
]

# Colonnes à exclure dans tous les cas
always_exclude_testfree = [
    'missing_avg_dns_time', 'missing_std_dns_time',
    'missing_avg_latence_scoring', 'missing_std_latence_scoring',
    'missing_avg_score_scoring', 'missing_std_score_scoring',
    'code_departement', 'date_hour', 'peag_nro',
    'boucle', 'olt_name'
]

# Colonnes catégorielles définies
categorical_cols_testfree = [
    'olt_model', 'dsp', 'pop_dns',
    'moment_journee', 'depts_egaux', 'meme_region',
    'type_boucle', 'type_olt_model', 'week_end'
]

# Toutes les colonnes possibles (y compris les colonnes de test)
default_features_testfree = [
    'olt_model', 'dsp', 'pop_dns', 'nb_client_total',
    'moment_journee', 'depts_egaux', 'meme_region',
    'type_boucle', 'type_olt_model',
    'encodage_heure_sin', 'encodage_heure_cos',
    'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos',
    'week_end',
    'nb_test_dns', 'avg_dns_time', 'std_dns_time',
    'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring',
    'avg_score_scoring', 'std_score_scoring'
]

# Exclusions spécifiques par cible (basé sur ton tableau)
target_specific_exclude_testfree = {
    'is_jump_avg_dns_time': [
        'nb_test_dns', 'avg_dns_time', 'std_dns_time',
        'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring',
        'avg_score_scoring', 'std_score_scoring'
    ],
    'is_jump_avg_latence_scoring': [
        'nb_test_dns', 'avg_dns_time', 'std_dns_time',
        'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring',
        'avg_score_scoring', 'std_score_scoring'
    ],
    'is_jump_avg_score_scoring': [
        'nb_test_dns', 'avg_dns_time', 'std_dns_time',
        'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring',
        'avg_score_scoring', 'std_score_scoring'
    ],
    'is_anomaly_dns': [
        'nb_test_dns', 'avg_dns_time', 'std_dns_time',
        'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring',
        'avg_score_scoring', 'std_score_scoring'
    ],
    'is_anomaly_latence': [
        'nb_test_dns', 'avg_dns_time', 'std_dns_time',
        'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring',
        'avg_score_scoring', 'std_score_scoring'
    ],
    'is_anomaly_scoring': [
        'nb_test_dns', 'avg_dns_time', 'std_dns_time',
        'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring',
        'avg_score_scoring', 'std_score_scoring'
    ]
}

# Fonction pour préparer les colonnes (version test-free)
def get_features_and_semantics_testfree(df, target_col):
    excluded = set(always_exclude_testfree + target_specific_exclude_testfree[target_col] + [col for col in target_cols if col != target_col])
    selected = [col for col in default_features_testfree if col not in excluded and col in df.columns]

    df_model = df[selected + [target_col]].dropna(subset=[target_col]).copy()
    df_model["target"] = df_model[target_col].astype(int)
    df_model.drop(columns=[target_col], inplace=True)

    print(f"\n--- Target : {target_col} ---")
    print("Colonnes gardées :", list(df_model.columns))

    semantics = []
    for col in df_model.columns:
        if col == "target":
            continue
        elif col in categorical_cols_testfree:
            semantics.append((col, Semantic.CATEGORICAL))
        else:
            semantics.append((col, Semantic.NUMERICAL))

    return df_model, semantics

# Fonction d'entraînement et d'affichage du modèle
def build_and_evaluate_cart_testfree(df_model, feature_semantics, label, title):
    print(f"\n=== Entraînement modèle : {title} ===")
    train_df, test_df = train_test_split(df_model, test_size=0.2, stratify=df_model[label], random_state=42)

    learner = CartLearner(
        label=label,
        task=Task.CLASSIFICATION,
        features=feature_semantics,
        include_all_columns=True,
        max_depth=5
    )

    model = learner.train(train_df)
    model.describe()
    evaluation = model.evaluate(test_df, label=label)
    print(evaluation)
    display(model.plot_tree())
    print("-" * 80)

# Lancement pour OLT
print("== Entraînement modèles OLT - TEST-FREE ==")
for target_col in target_cols:
    df_model, semantics = get_features_and_semantics_testfree(df_olt, target_col)
    build_and_evaluate_cart_testfree(df_model, semantics, "target", f"OLT - {target_col} (test-free)")

# Lancement pour PEAG
print("== Entraînement modèles PEAG - TEST-FREE ==")
for target_col in target_cols:
    df_model, semantics = get_features_and_semantics_testfree(df_peag, target_col)
    build_and_evaluate_cart_testfree(df_model, semantics, "target", f"PEAG - {target_col} (test-free)")


== Entraînement modèles OLT - TEST-FREE ==

--- Target : is_jump_avg_dns_time ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'target']

=== Entraînement modèle : OLT - is_jump_avg_dns_time (test-free) ===
Train model on 2973073 examples
Model trained in 0:00:03.526438
accuracy: 0.993838
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 738689 |      0 |
    +--------+--------+--------+
    |      1 |   4580 |      0 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.5
    PR AUC: 0.503081
    Num thresholds: 3
loss: 0.0375034
num examples: 743269
num examples (weighted): 743269



--------------------------------------------------------------------------------

--- Target : is_jump_avg_latence_scoring ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'target']

=== Entraînement modèle : OLT - is_jump_avg_latence_scoring (test-free) ===
Train model on 2973073 examples
Model trained in 0:00:03.510772
accuracy: 0.993391
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 738357 |      0 |
    +--------+--------+--------+
    |      1 |   4912 |      0 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.5
    PR AUC: 0.503304
    Num thresholds: 3
loss: 0.039758
num examples: 743269
num examples (wei

--------------------------------------------------------------------------------

--- Target : is_jump_avg_score_scoring ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'target']

=== Entraînement modèle : OLT - is_jump_avg_score_scoring (test-free) ===
Train model on 2973073 examples
Model trained in 0:00:03.543255
accuracy: 0.994378
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 739090 |      0 |
    +--------+--------+--------+
    |      1 |   4179 |      0 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.5
    PR AUC: 0.502811
    Num thresholds: 3
loss: 0.0347365
num examples: 743269
num examples (weight

--------------------------------------------------------------------------------

--- Target : is_anomaly_dns ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'target']

=== Entraînement modèle : OLT - is_anomaly_dns (test-free) ===
Train model on 2973073 examples
Model trained in 0:00:03.453121
accuracy: 0.994106
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 738888 |      0 |
    +--------+--------+--------+
    |      1 |   4381 |      0 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.5
    PR AUC: 0.502947
    Num thresholds: 3
loss: 0.0361365
num examples: 743269
num examples (weighted): 743269



--------------------------------------------------------------------------------

--- Target : is_anomaly_latence ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'target']

=== Entraînement modèle : OLT - is_anomaly_latence (test-free) ===
Train model on 2973073 examples
Model trained in 0:00:03.633457
accuracy: 0.99522
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 739716 |      0 |
    +--------+--------+--------+
    |      1 |   3553 |      0 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.5
    PR AUC: 0.50239
    Num thresholds: 3
loss: 0.0303109
num examples: 743269
num examples (weighted): 743269



...

--------------------------------------------------------------------------------

--- Target : is_jump_avg_score_scoring ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'target']

=== Entraînement modèle : PEAG - is_jump_avg_score_scoring (test-free) ===
Train model on 2973073 examples
Model trained in 0:00:03.615979
accuracy: 0.994378
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 739090 |      0 |
    +--------+--------+--------+
    |      1 |   4179 |      0 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.5
    PR AUC: 0.502811
    Num thresholds: 3
loss: 0.0347365
num examples: 743269
num examples (weigh

--------------------------------------------------------------------------------

--- Target : is_anomaly_dns ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'target']

=== Entraînement modèle : PEAG - is_anomaly_dns (test-free) ===
Train model on 2973073 examples
Model trained in 0:00:03.512799
accuracy: 0.994106
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 738888 |      0 |
    +--------+--------+--------+
    |      1 |   4381 |      0 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.5
    PR AUC: 0.502947
    Num thresholds: 3
loss: 0.0361365
num examples: 743269
num examples (weighted): 743269



--------------------------------------------------------------------------------

--- Target : is_anomaly_latence ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'target']

=== Entraînement modèle : PEAG - is_anomaly_latence (test-free) ===
Train model on 2973073 examples
Model trained in 0:00:03.503706
accuracy: 0.99522
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 739716 |      0 |
    +--------+--------+--------+
    |      1 |   3553 |      0 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.5
    PR AUC: 0.50239
    Num thresholds: 3
loss: 0.0303109
num examples: 743269
num examples (weighted): 743269



--------------------------------------------------------------------------------

--- Target : is_anomaly_scoring ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'target']

=== Entraînement modèle : PEAG - is_anomaly_scoring (test-free) ===
Train model on 2973073 examples
Model trained in 0:00:03.514934
accuracy: 0.998624
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 742246 |      0 |
    +--------+--------+--------+
    |      1 |   1023 |      0 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.5
    PR AUC: 0.500688
    Num thresholds: 3
loss: 0.0104433
num examples: 743269
num examples (weighted): 743269



--------------------------------------------------------------------------------


## Gradient boosting & interprétabilité : AVEC TESTS ##

In [13]:
import pandas as pd
import numpy as np
import ydf
from ydf import Task
from sklearn.model_selection import train_test_split
from IPython.display import display

# Chargement des fichiers
dataset_olt = '/content/drive/MyDrive/nexialog/data_with_anomalies_OLT_with_jump_and_DB_SCAN.parquet'
dataset_peag = "/content/drive/MyDrive/nexialog/data_with_anomalies_PEAG_with_jump_and_DB_SCAN.parquet"
df_olt = pd.read_parquet(dataset_olt)
df_peag = pd.read_parquet(dataset_peag)

# Cibles
target_cols = [
    'is_jump_avg_dns_time', 'is_jump_avg_latence_scoring', 'is_jump_avg_score_scoring',
    'is_anomaly_dns', 'is_anomaly_latence', 'is_anomaly_scoring'
]

# Colonnes à exclure dans tous les cas
always_exclude = [
    'missing_avg_dns_time', 'missing_std_dns_time',
    'missing_avg_latence_scoring', 'missing_std_latence_scoring',
    'missing_avg_score_scoring', 'missing_std_score_scoring',
    'code_departement', 'date_hour', 'peag_nro',
    'boucle', 'olt_name'
]

# Colonnes gardées par défaut
default_features = [
    'olt_model', 'dsp', 'pop_dns', 'nb_client_total',
    'moment_journee', 'depts_egaux', 'meme_region',
    'type_boucle', 'type_olt_model',
    'encodage_heure_sin', 'encodage_heure_cos',
    'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos',
    'week_end',
    'nb_test_dns', 'avg_dns_time', 'std_dns_time',
    'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring',
    'avg_score_scoring', 'std_score_scoring'
]

# Cibles spécifiques à exclure par target
target_specific_exclude = {
    'is_jump_avg_dns_time': ['nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring', 'avg_score_scoring', 'std_score_scoring'],
    'is_jump_avg_latence_scoring': ['nb_test_dns', 'avg_dns_time', 'std_dns_time', 'avg_score_scoring', 'std_score_scoring'],
    'is_jump_avg_score_scoring': ['nb_test_dns', 'avg_dns_time', 'std_dns_time', 'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring'],
    'is_anomaly_dns': ['nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring', 'avg_score_scoring', 'std_score_scoring'],
    'is_anomaly_latence': ['nb_test_dns', 'avg_dns_time', 'std_dns_time', 'avg_score_scoring', 'std_score_scoring'],
    'is_anomaly_scoring': ['nb_test_dns', 'avg_dns_time', 'std_dns_time', 'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring']
}

def train_gbt(df, dataset_name):
    for target_col in target_cols:
        print(f"\n=== Entraînement modèle GBT {dataset_name} - {target_col} ===")

        all_excluded = set(always_exclude + target_specific_exclude.get(target_col, []) + [col for col in target_cols if col != target_col])
        selected_cols = [col for col in default_features if col not in all_excluded and col in df.columns]

        df_model = df[selected_cols + [target_col]].dropna(subset=[target_col]).copy()
        df_model[target_col] = df_model[target_col].astype(int)

        print("Colonnes gardées :", df_model.columns.tolist())

        train_df, test_df = train_test_split(df_model, test_size=0.2, stratify=df_model[target_col], random_state=42)

        model = ydf.GradientBoostedTreesLearner(label=target_col, task=Task.CLASSIFICATION).train(train_df)
        display(model.describe())
        evaluation = model.evaluate(test_df)
        print("Test accuracy:", evaluation.accuracy)
        print("Rapport complet :")
        print(evaluation)
        display(model.analyze(test_df, sampling=1))

print("=== Entraînement GBT - OLT ===")
train_gbt(df_olt, "OLT")

print("=== Entraînement GBT - PEAG ===")
train_gbt(df_peag, "PEAG")


=== Entraînement GBT - OLT ===

=== Entraînement modèle GBT OLT - is_jump_avg_dns_time ===
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'nb_test_dns', 'avg_dns_time', 'std_dns_time', 'is_jump_avg_dns_time']
Train model on 2973073 examples
Model trained in 0:02:48.601589


Test accuracy: 0.9970239576788484
Rapport complet :
accuracy: 0.997024
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 737707 |    982 |
    +--------+--------+--------+
    |      1 |   1230 |   3350 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.994562
    PR AUC: 0.795296
    Num thresholds: 10000
loss: 0.00872962
num examples: 743269
num examples (weighted): 743269




=== Entraînement modèle GBT OLT - is_jump_avg_latence_scoring ===
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring', 'is_jump_avg_latence_scoring']
Train model on 2973073 examples
Model trained in 0:03:11.895040


Test accuracy: 0.9949816284548394
Rapport complet :
accuracy: 0.994982
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 737320 |   1037 |
    +--------+--------+--------+
    |      1 |   2693 |   2219 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.990002
    PR AUC: 0.611706
    Num thresholds: 10000
loss: 0.0135487
num examples: 743269
num examples (weighted): 743269




=== Entraînement modèle GBT OLT - is_jump_avg_score_scoring ===
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'avg_score_scoring', 'std_score_scoring', 'is_jump_avg_score_scoring']
Train model on 2973073 examples
Model trained in 0:03:55.389701


...

Test accuracy: 0.9972284596828336
Rapport complet :
accuracy: 0.997228
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 738150 |    738 |
    +--------+--------+--------+
    |      1 |   1322 |   3059 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.991182
    PR AUC: 0.815686
    Num thresholds: 10000
loss: 0.00848641
num examples: 743269
num examples (weighted): 743269




=== Entraînement modèle GBT PEAG - is_anomaly_latence ===
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring', 'is_anomaly_latence']
Train model on 2973073 examples
Model trained in 0:03:02.002267


Test accuracy: 0.9969055617818045
Rapport complet :
accuracy: 0.996906
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 739145 |    571 |
    +--------+--------+--------+
    |      1 |   1729 |   1824 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.991165
    PR AUC: 0.682162
    Num thresholds: 10000
loss: 0.00946834
num examples: 743269
num examples (weighted): 743269




=== Entraînement modèle GBT PEAG - is_anomaly_scoring ===
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'avg_score_scoring', 'std_score_scoring', 'is_anomaly_scoring']
Train model on 2973073 examples
Model trained in 0:03:30.641806


Test accuracy: 0.9986209568810216
Rapport complet :
accuracy: 0.998621
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 742179 |     67 |
    +--------+--------+--------+
    |      1 |    958 |     65 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.942308
    PR AUC: 0.139857
    Num thresholds: 10000
loss: 0.00701042
num examples: 743269
num examples (weighted): 743269



## Gradient boosting & interprétabilité : TEST-FREE ##

In [15]:
import pandas as pd
import numpy as np
import ydf
from ydf import Task, Semantic
from sklearn.model_selection import train_test_split
from IPython.display import display

# === Chargement des fichiers ===
dataset_olt = '/content/drive/MyDrive/nexialog/data_with_anomalies_OLT_with_jump_and_DB_SCAN.parquet'
dataset_peag = "/content/drive/MyDrive/nexialog/data_with_anomalies_PEAG_with_jump_and_DB_SCAN.parquet"
df_olt = pd.read_parquet(dataset_olt)
df_peag = pd.read_parquet(dataset_peag)

# === Définition des cibles ===
target_cols = [
    'is_jump_avg_dns_time', 'is_jump_avg_latence_scoring', 'is_jump_avg_score_scoring',
    'is_anomaly_dns', 'is_anomaly_latence', 'is_anomaly_scoring'
]

# === Exclusions fixes ===
always_exclude_testfree = [
    'missing_avg_dns_time', 'missing_std_dns_time',
    'missing_avg_latence_scoring', 'missing_std_latence_scoring',
    'missing_avg_score_scoring', 'missing_std_score_scoring',
    'code_departement', 'date_hour', 'peag_nro', 'boucle', 'olt_name'
]

# === Variables catégorielles ===
categorical_cols_testfree = [
    'olt_model', 'dsp', 'pop_dns',
    'moment_journee', 'depts_egaux', 'meme_region',
    'type_boucle', 'type_olt_model', 'week_end'
]

# === Toutes les features disponibles (y compris les tests) ===
default_features_testfree = [
    'olt_model', 'dsp', 'pop_dns', 'nb_client_total',
    'moment_journee', 'depts_egaux', 'meme_region',
    'type_boucle', 'type_olt_model',
    'encodage_heure_sin', 'encodage_heure_cos',
    'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos',
    'week_end',
    'nb_test_dns', 'avg_dns_time', 'std_dns_time',
    'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring',
    'avg_score_scoring', 'std_score_scoring'
]

# === Exclusions très strictes : toutes les colonnes de test ===
target_specific_exclude_testfree = {
    target: [
        'nb_test_dns', 'avg_dns_time', 'std_dns_time',
        'nb_test_scoring', 'avg_latence_scoring', 'std_latence_scoring',
        'avg_score_scoring', 'std_score_scoring'
    ] for target in target_cols
}

def get_features_and_semantics_testfree(df, target_col):
    excluded = set(always_exclude_testfree + target_specific_exclude_testfree[target_col] + [
        col for col in target_cols if col != target_col
    ])
    selected = [col for col in default_features_testfree if col not in excluded and col in df.columns]

    # On garde target_col comme label, mais pas comme feature
    df_model = df[selected + [target_col]].dropna(subset=[target_col]).copy()
    df_model["target"] = df_model[target_col].astype(int)
    df_model.drop(columns=[target_col], inplace=True)  # ❗ remove original target_col to avoid collision

    print(f"\n--- Target : {target_col} ---")
    print("Colonnes gardées :", list(df_model.columns))

    semantics = []
    for col in df_model.columns:
        if col == "target":
            continue
        elif col in categorical_cols_testfree:
            semantics.append((col, Semantic.CATEGORICAL))
        else:
            semantics.append((col, Semantic.NUMERICAL))

    return df_model, semantics


def build_and_evaluate_gbt_testfree(df_model, feature_semantics, label, title):
    print(f"\n=== Entraînement GBT : {title} ===")
    train_df, test_df = train_test_split(df_model, test_size=0.2, stratify=df_model[label], random_state=42)

    model = ydf.GradientBoostedTreesLearner(
        label=label,
        task=Task.CLASSIFICATION,
        features=feature_semantics,
        include_all_columns=True
    ).train(train_df)

    display(model.describe())
    evaluation = model.evaluate(test_df)
    print("Test accuracy:", evaluation.accuracy)
    print("Rapport complet :")
    print(evaluation)
    display(model.analyze(test_df, sampling=1))

# === Entraînement sur OLT ===
print("== Entraînement GBT TEST-FREE - OLT ==")
for target_col in target_cols:
    df_model, semantics = get_features_and_semantics_testfree(df_olt, target_col)
    build_and_evaluate_gbt_testfree(df_model, semantics, "target", f"OLT - {target_col} (test-free)")

# === Entraînement sur PEAG ===
print("== Entraînement GBT TEST-FREE - PEAG ==")
for target_col in target_cols:
    df_model, semantics = get_features_and_semantics_testfree(df_peag, target_col)
    build_and_evaluate_gbt_testfree(df_model, semantics, "target", f"PEAG - {target_col} (test-free)")


== Entraînement GBT TEST-FREE - OLT ==

--- Target : is_jump_avg_dns_time ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'target']

=== Entraînement GBT : OLT - is_jump_avg_dns_time (test-free) ===
Train model on 2973073 examples
Model trained in 0:01:12.012735


Test accuracy: 0.993839377129949
Rapport complet :
accuracy: 0.993839
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 738686 |      3 |
    +--------+--------+--------+
    |      1 |   4576 |      4 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.649042
    PR AUC: 0.0129608
    Num thresholds: 10000
loss: 0.0365905
num examples: 743269
num examples (weighted): 743269




--- Target : is_jump_avg_latence_scoring ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'target']

=== Entraînement GBT : OLT - is_jump_avg_latence_scoring (test-free) ===
Train model on 2973073 examples
Model trained in 0:01:11.522625


Test accuracy: 0.993391356292271
Rapport complet :
accuracy: 0.993391
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 738357 |      0 |
    +--------+--------+--------+
    |      1 |   4912 |      0 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.597489
    PR AUC: 0.00958275
    Num thresholds: 10000
loss: 0.0393461
num examples: 743269
num examples (weighted): 743269




--- Target : is_jump_avg_score_scoring ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'target']

=== Entraînement GBT : OLT - is_jump_avg_score_scoring (test-free) ===
Train model on 2973073 examples
Model trained in 0:03:42.789638


...

Test accuracy: 0.9942483811379191
Rapport complet :
accuracy: 0.994248
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 738831 |     57 |
    +--------+--------+--------+
    |      1 |   4218 |    163 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.655326
    PR AUC: 0.0650546
    Num thresholds: 10000
loss: 0.0339937
num examples: 743269
num examples (weighted): 743269




--- Target : is_anomaly_latence ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'target']

=== Entraînement GBT : PEAG - is_anomaly_latence (test-free) ===
Train model on 2973073 examples
Model trained in 0:01:12.492735


Test accuracy: 0.9952197656568483
Rapport complet :
accuracy: 0.99522
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 739716 |      0 |
    +--------+--------+--------+
    |      1 |   3553 |      0 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.585173
    PR AUC: 0.00774225
    Num thresholds: 10000
loss: 0.030021
num examples: 743269
num examples (weighted): 743269




--- Target : is_anomaly_scoring ---
Colonnes gardées : ['olt_model', 'dsp', 'pop_dns', 'nb_client_total', 'moment_journee', 'depts_egaux', 'meme_region', 'type_boucle', 'type_olt_model', 'encodage_heure_sin', 'encodage_heure_cos', 'encodage_jour_semaine_sin', 'encodage_jour_semaine_cos', 'week_end', 'target']

=== Entraînement GBT : PEAG - is_anomaly_scoring (test-free) ===
Train model on 2973073 examples
Model trained in 0:02:31.086449


Test accuracy: 0.9986209568810216
Rapport complet :
accuracy: 0.998621
confusion matrix:
    label (row) \ prediction (col)
    +--------+--------+--------+
    |        |      0 |      1 |
    +--------+--------+--------+
    |      0 | 742244 |      2 |
    +--------+--------+--------+
    |      1 |   1023 |      0 |
    +--------+--------+--------+
characteristics:
    name: '1' vs others
    ROC AUC: 0.781654
    PR AUC: 0.0060221
    Num thresholds: 10000
loss: 0.00968538
num examples: 743269
num examples (weighted): 743269



## EXPLORATIVE ANALYSIS ##

In [3]:
import pandas as pd
import plotly.graph_objects as go

dataset_olt = '/content/drive/MyDrive/nexialog/data_with_anomalies_OLT_with_jump_and_DB_SCAN.parquet'
dataset_peag = "/content/drive/MyDrive/nexialog/data_with_anomalies_PEAG_with_jump_and_DB_SCAN.parquet"
df_olt = pd.read_parquet(dataset_olt)
df_peag = pd.read_parquet(dataset_peag)


In [23]:
import plotly.graph_objects as go

target = 'is_jump_avg_dns_time'

# Données préparées
anomalies_by_model = df_olt.groupby("olt_model")[target].sum().reset_index()
anomalies_by_model.columns = ['olt_model', 'nb_anomalies']
anomalies_by_model = anomalies_by_model.sort_values(by='nb_anomalies', ascending=False)
anomalies_by_model['pct_cumul_anomalies'] = anomalies_by_model['nb_anomalies'].cumsum() / anomalies_by_model['nb_anomalies'].sum() * 100

# Graphique
fig = go.Figure()

# Barres
fig.add_trace(go.Bar(
    x=anomalies_by_model['olt_model'],
    y=anomalies_by_model['nb_anomalies'],
    name="Nombre d'anomalies",
    marker_color='rgba(55, 83, 109, 0.8)',
    text=anomalies_by_model['nb_anomalies'],
    textposition='auto'
))

# Courbe % cumulé
fig.add_trace(go.Scatter(
    x=anomalies_by_model['olt_model'],
    y=anomalies_by_model['pct_cumul_anomalies'],
    name="Pourcentage cumulé des anomalies",
    yaxis='y2',
    mode='lines+markers',
    line=dict(color='royalblue', width=2)
))

# Layout clair et responsive
fig.update_layout(
    title={
        'text': f"<b>Concentration des anomalies pour {target}</b>",
        'x': 0.5,
        'xanchor': 'center',
        'font': dict(size=20)
    },
    xaxis=dict(title="Modèle OLT", tickangle=-45),
    yaxis=dict(title="Nombre d'anomalies"),
    yaxis2=dict(title="Pourcentage cumulé des anomalies", overlaying='y', side='right', showgrid=False),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=60, r=60, t=80, b=100),
    height=500,
    template="plotly_white"
)

fig.show()


In [25]:

# 1. Nombre d'anomalies par modèle OLT
anomalies_by_model = df_olt.groupby("olt_model")[target].sum().reset_index()
anomalies_by_model.columns = ['olt_model', 'nb_anomalies']
anomalies_by_model['pct_anomalies'] = anomalies_by_model['nb_anomalies'] / anomalies_by_model['nb_anomalies'].sum() * 100

# 2. Part de chaque modèle dans le dataset
presence_by_model = df_olt['olt_model'].value_counts(normalize=True).reset_index()
presence_by_model.columns = ['olt_model', 'pct_presence']
presence_by_model['pct_presence'] *= 100

# 3. Fusion et tri
merged = pd.merge(anomalies_by_model, presence_by_model, on='olt_model')
merged = merged.sort_values(by='pct_anomalies', ascending=False)

display(merged)

,olt_model,nb_anomalies,pct_anomalies,pct_presence
3,M22,8449,36.893585,42.316584
1,M13,5800,25.326405,19.142049
0,M11,3868,16.890092,17.831916
5,old0,2235,9.759399,9.298337
4,M24,1812,7.912318,7.122864
6,old1,561,2.449675,3.454714
2,M20,176,0.768525,0.833535


In [26]:


target = "is_jump_avg_dns_time"

summary = df_olt.groupby("week_end").agg(
    nb_observations=('week_end', 'count'),
    nb_anomalies=(target, 'sum')
).reset_index()

summary['taux_anomalies (%)'] = (summary['nb_anomalies'] / summary['nb_observations'] * 100).round(2)
summary['week_end_label'] = summary['week_end'].map({False: 'Semaine', True: 'Week-end'})

fig = px.bar(
    summary,
    x='week_end_label',
    y='taux_anomalies (%)',
    text='taux_anomalies (%)',
    labels={'week_end_label': 'Jour', 'taux_anomalies (%)': 'Taux d\'anomalie (%)'},
    title=f"Taux d'anomalies ({target}) - Semaine vs Week-end"
)
fig.update_traces(textposition='outside')
fig.update_layout(yaxis_range=[0, 100])

fig.show()


In [27]:


# Nettoyage : on enlève les éventuels NaN
df_filtered = df_olt.dropna(subset=["is_jump_avg_dns_time", "moment_journee"]).copy()
df_filtered["is_jump_avg_dns_time"] = df_filtered["is_jump_avg_dns_time"].astype(int)

# Calcul du taux d'anomalie par moment de la journée
grouped = df_filtered.groupby("moment_journee").agg(
    total=('is_jump_avg_dns_time', 'count'),
    anomalies=('is_jump_avg_dns_time', 'sum')
).reset_index()
grouped["taux_anomalie"] = (grouped["anomalies"] / grouped["total"]) * 100

# Tri pour affichage clair
grouped = grouped.sort_values("taux_anomalie", ascending=False)

# Plot
fig = px.bar(
    grouped,
    x="moment_journee",
    y="taux_anomalie",
    text=grouped["taux_anomalie"].round(2),
    labels={"moment_journee": "Moment de la journée", "taux_anomalie": "Taux d'anomalie (%)"},
    title="Taux d'anomalies (is_jump_avg_dns_time) selon le moment de la journée"
)
fig.update_traces(textposition='outside')
fig.update_layout(yaxis_range=[0, grouped["taux_anomalie"].max() * 1.2])

fig.show()

In [63]:



# Nombre d’anomalies et volume total par moment
anomalies = df_olt[df_olt[target_col] == 1]['moment_journee'].value_counts()
totaux = df_olt['moment_journee'].value_counts()

# Mise en forme
result = pd.DataFrame({'nb_anomalies': anomalies, 'nb_total': totaux}).fillna(0)
result['pct_anomalies'] = 100 * result['nb_anomalies'] / result['nb_anomalies'].sum()
result['pct_trafic'] = 100 * result['nb_total'] / result['nb_total'].sum()
result = result.reset_index().rename(columns={'index': 'moment_journee'})

# Tri logique
ordre = ['matin', 'après-midi', 'soir', 'nuit']
result['moment_journee'] = pd.Categorical(result['moment_journee'], categories=ordre, ordered=True)
result = result.sort_values('moment_journee')

fig = go.Figure()

# Barres : anomalies
fig.add_trace(go.Bar(
    x=result['moment_journee'],
    y=result['pct_anomalies'],
    name="% des anomalies",
    text=[f"{v:.1f}%" for v in result['pct_anomalies']],
    textposition='outside',
    marker_color='crimson',
    hovertemplate='Anomalies : %{y:.1f}%<extra></extra>'
))

# Courbe : trafic
fig.add_trace(go.Scatter(
    x=result['moment_journee'],
    y=result['pct_trafic'],
    name="% du trafic",
    mode='lines+markers+text',
    text=[f"{v:.2f}%" for v in result['pct_trafic']],
    textposition='top right',
    textfont=dict(size=12),
    line=dict(color='royalblue', width=1),
    marker=dict(size=6),
    hovertemplate='Trafic : %{y:.1f}%<extra></extra>'
))

fig.update_layout(
    title=f"<b>Répartition des anomalies et du trafic par moment de la journée</b><br><sup>{target_col}</sup>",
    xaxis_title="Moment de la journée",
    yaxis_title="Pourcentage (%)",
    yaxis=dict(range=[0, 40], tickformat=".2f"),
    bargap=0.3,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
)

fig.show()


In [40]:
# Nombre total de lignes et anomalies par OLT
olt_counts = df_olt.groupby('olt_name').agg(
    nb_total=('is_jump_avg_dns_time', 'count'),
    nb_anomalies=('is_jump_avg_dns_time', 'sum')
).reset_index()

# Ajout du taux d’anomalie
olt_counts['taux_anomalie'] = olt_counts['nb_anomalies'] / olt_counts['nb_total']

# On garde les OLT avec un minimum de volume (ex: 100)
filtered_olt = olt_counts[olt_counts['nb_total'] >= 100]

# Top 30 par taux d’anomalie
top_olts_taux = filtered_olt.sort_values('taux_anomalie', ascending=False).head(30)

# Graphique
fig = px.bar(
    top_olts_taux,
    x='olt_name',
    y='taux_anomalie',
    title="Top 30 des OLT avec le plus fort taux d'anomalie (≥ 100 lignes)",
    labels={'olt_name': 'OLT', 'taux_anomalie': "Taux d'anomalie"},
    text=top_olts_taux['taux_anomalie'].apply(lambda x: f"{x:.1%}")
)

fig.update_traces(marker_color='darkorange', textposition='outside')
fig.update_layout(xaxis_tickangle=-45, template='plotly_white')
fig.show()


In [41]:
# Comptage des lignes par OLT
olt_counts = df_olt['olt_name'].value_counts()

# Méthode 1 : seuil = 75e percentile
seuil_percentile = int(olt_counts.quantile(0.75))
olts_valides_percentile = olt_counts[olt_counts >= seuil_percentile].index.tolist()
df_filtre_percentile = df_olt[df_olt['olt_name'].isin(olts_valides_percentile)]

# Méthode 2 : Pareto - on garde les OLT qui couvrent 80% du volume
olt_counts_sorted = olt_counts.sort_values(ascending=False)
cumulative = olt_counts_sorted.cumsum()
total = cumulative.max()
olts_valides_pareto = cumulative[cumulative <= 0.8 * total].index.tolist()
df_filtre_pareto = df_olt[df_olt['olt_name'].isin(olts_valides_pareto)]

# Affichage des comparaisons
print("=== Méthode 1 : Seuil Percentile ===")
print(f"Seuil retenu : {seuil_percentile} lignes minimum")
print(f"OLT conservés : {len(olts_valides_percentile)}")
print(f"Lignes totales conservées : {len(df_filtre_percentile)}")

print("\n=== Méthode 2 : Pareto 80% ===")
print(f"OLT conservés : {len(olts_valides_pareto)}")
print(f"Lignes totales conservées : {len(df_filtre_pareto)}")


=== Méthode 1 : Seuil Percentile ===
Seuil retenu : 1436 lignes minimum
OLT conservés : 999
Lignes totales conservées : 1437242

=== Méthode 2 : Pareto 80% ===
OLT conservés : 2156
Lignes totales conservées : 2972146


In [44]:
# Liste des cibles
target_cols = [
    'is_jump_avg_dns_time'
]

for target in target_cols:
    print(f"=== Graphique pour : {target} ===")

    # Nombre total de lignes et anomalies par OLT
    olt_counts = df_olt.groupby('olt_name').agg(
        nb_total=(target, 'count'),
        nb_anomalies=(target, 'sum')
    ).reset_index()

    #  Ajout du taux d’anomalie
    olt_counts['taux_anomalie'] = olt_counts['nb_anomalies'] / olt_counts['nb_total']

    # Méthode Pareto : on garde les OLT représentant 80 % du volume total
    olt_counts = olt_counts.sort_values('nb_total', ascending=False)
    olt_counts['cumul'] = olt_counts['nb_total'].cumsum()
    seuil_total = olt_counts['nb_total'].sum() * 0.80
    filtered_olt = olt_counts[olt_counts['cumul'] <= seuil_total]

    #  Top 30 par taux d’anomalie
    top_olts_taux = filtered_olt.sort_values('taux_anomalie', ascending=False).head(30)

    # Graphique
    fig = px.bar(
        top_olts_taux,
        x='olt_name',
        y='taux_anomalie',
        title=f"Top 30 OLT – Plus fort taux d’anomalie ({target}) [Méthode Pareto – 80% du trafic]",
        labels={'olt_name': 'OLT', 'taux_anomalie': "Taux d'anomalie"},
        text=top_olts_taux['taux_anomalie'].apply(lambda x: f"{x:.1%}")
    )

    fig.update_traces(marker_color='darkorange', textposition='outside')
    fig.update_layout(xaxis_tickangle=-45, template='plotly_white')
    fig.show()


=== Graphique pour : is_jump_avg_dns_time ===


In [51]:
#  Agrégation par OLT sur la cible en cours
target = 'is_jump_avg_dns_time'
olt_counts = df_olt.groupby('olt_name').agg(
    nb_total=(target, 'count'),
    nb_anomalies=(target, 'sum')
).reset_index()

#  Taux d’anomalie
olt_counts['taux_anomalie'] = olt_counts['nb_anomalies'] / olt_counts['nb_total']

#  Moyenne sur tous les OLT
mean_taux_anomalie = olt_counts['taux_anomalie'].mean()

#  Calcul du seuil Pareto (80% des lignes conservées)
olt_sorted = olt_counts.sort_values('nb_total', ascending=False).copy()
olt_sorted['cumsum'] = olt_sorted['nb_total'].cumsum()
total_lignes = olt_sorted['nb_total'].sum()
olt_sorted['cumsum_pct'] = olt_sorted['cumsum'] / total_lignes

olt_pareto = olt_sorted[olt_sorted['cumsum_pct'] <= 0.8]
seuil_pareto = olt_pareto['nb_total'].min()

#  Minimum dans les OLT au-dessus de ce seuil
filtered_olt = olt_counts[olt_counts['nb_total'] >= seuil_pareto]
min_taux_anomalie = filtered_olt['taux_anomalie'].min()
olt_min = filtered_olt[filtered_olt['taux_anomalie'] == min_taux_anomalie]

# 📤 Résultat
print(f" Moyenne globale (tous OLT) : {mean_taux_anomalie:.2%}")
print(f" Plus faible taux (parmi OLT ≥ {seuil_pareto} lignes) : {min_taux_anomalie:.2%}")
print(olt_min[['olt_name', 'taux_anomalie', 'nb_total']])



🎯 Moyenne globale (tous OLT) : 0.57%
🔽 Plus faible taux (parmi OLT ≥ 1048 lignes) : 0.00%
         olt_name  taux_anomalie  nb_total
195    07_olt_299            0.0      1355
768   26_olt_1291            0.0      1390
775   26_olt_1299            0.0      1371
776   26_olt_1300            0.0      1399
1267  35_olt_1992            0.0      1215
1636  44_olt_2596            0.0      1141
2088  57_olt_3280            0.0      1405
2640  68_olt_3979            0.0      1298
3541  87_olt_5257            0.0      1439


In [52]:


target_col = 'is_jump_avg_dns_time'

#  Agrégation par type d’OLT
type_model_stats = df_olt.groupby('type_olt_model').agg(
    nb_total=(target_col, 'count'),
    nb_anomalies=(target_col, 'sum')
).reset_index()
type_model_stats['taux_anomalie'] = type_model_stats['nb_anomalies'] / type_model_stats['nb_total']

#  Moyenne globale (référence)
moyenne_globale = df_olt[target_col].mean()

#  Tri décroissant
type_model_stats = type_model_stats.sort_values('taux_anomalie', ascending=False)

#  Graphique
fig = px.bar(
    type_model_stats,
    x='type_olt_model',
    y='taux_anomalie',
    title=f"Taux d’anomalie moyen par type d’OLT - {target_col}",
    labels={'type_olt_model': 'Type OLT', 'taux_anomalie': "Taux d'anomalie"},
    text=type_model_stats['taux_anomalie'].apply(lambda x: f"{x:.2%}")
)

fig.add_hline(y=moyenne_globale, line_dash='dash', line_color='black',
              annotation_text=f"Moyenne globale : {moyenne_globale:.2%}",
              annotation_position="top left")

fig.update_traces(marker_color='teal', textposition='outside')
fig.update_layout(xaxis_tickangle=-45, template='plotly_white')
fig.show()


In [65]:


#  Sécurisation de l'heure (au cas où des NaT ou formats anormaux seraient présents)
df_olt = df_olt.copy()
df_olt['heure'] = pd.to_datetime(df_olt['date_hour'], errors='coerce').dt.hour

#  Agrégation par heure
heure_anomalie = df_olt.groupby('heure', dropna=True).agg(
    total=('is_jump_avg_dns_time', 'count'),
    anomalies=('is_jump_avg_dns_time', 'sum')
).reset_index()
heure_anomalie['taux_anomalie'] = heure_anomalie['anomalies'] / heure_anomalie['total']
heure_anomalie['text'] = heure_anomalie['taux_anomalie'].apply(lambda x: f"{x:.2%}")

#  Graphique
fig = px.line(
    heure_anomalie,
    x='heure',
    y='taux_anomalie',
    text='text',
    markers=True,
    title="Taux d’anomalie par heure de la journée – is_jump_avg_dns_time",
    labels={'heure': 'Heure', 'taux_anomalie': "Taux d'anomalie"}
)

fig.update_traces(line_color='teal', textposition='top center', marker=dict(color='darkcyan', size=7))
fig.update_layout(
    template='plotly_white',
    xaxis=dict(
        dtick=1,  # une heure par tick
        range=[0, 23]  # 🔒 verrouille l'axe de 0 à 23
    )
)


fig.show()


In [66]:
import pandas as pd

# Exemple simulé basé sur les taux du graphe (à adapter avec les vraies données si dispo)
taux_anomalie_par_heure = [
    0.0058, 0.0043, 0.0033, 0.0026, 0.0025, 0.0023,
    0.0033, 0.0044, 0.0060, 0.0072, 0.0072, 0.0080,
    0.0075, 0.0072, 0.0073, 0.0073, 0.0076, 0.0080,
    0.0087, 0.0070, 0.0078, 0.0092, 0.0071, 0.0061
]

# Calcul de la moyenne du taux d’anomalie horaire
moyenne_horaire = sum(taux_anomalie_par_heure) / len(taux_anomalie_par_heure)

# Récupérer le pic
pic = max(taux_anomalie_par_heure)

# Calcul du dépassement en pourcentage
surplus_pct = (pic - moyenne_horaire) / moyenne_horaire * 100

moyenne_horaire, pic, surplus_pct


(0.006154166666666666, 0.0092, 49.49221394719026)

In [60]:


#  Dictionnaire de traduction anglais → français
jours_fr = {
    'Monday': 'Lundi', 'Tuesday': 'Mardi', 'Wednesday': 'Mercredi',
    'Thursday': 'Jeudi', 'Friday': 'Vendredi', 'Saturday': 'Samedi', 'Sunday': 'Dimanche'
}

#  Extraction du jour
df_olt['jour_semaine'] = df_olt['date_hour'].dt.day_name().map(jours_fr)

#  Cible
target_col = 'is_jump_avg_dns_time'

#  Compter anomalies et volume
anomalies_jour = df_olt[df_olt[target_col] == 1]['jour_semaine'].value_counts().rename('nb_anomalies')
volume_jour = df_olt['jour_semaine'].value_counts().rename('nb_total')

#  DataFrame
df_jour = pd.concat([anomalies_jour, volume_jour], axis=1).fillna(0)
df_jour['taux_anomalie'] = df_jour['nb_anomalies'] / df_jour['nb_total']
df_jour = df_jour.reset_index().rename(columns={'index': 'jour_semaine'})

#  Ordre des jours
ordre = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche']
df_jour['jour_semaine'] = pd.Categorical(df_jour['jour_semaine'], categories=ordre, ordered=True)
df_jour = df_jour.sort_values('jour_semaine')

#  Ajout colonne texte pour annotation
df_jour['text'] = df_jour['taux_anomalie'].apply(lambda x: f"{x:.2%}")

#  Graphique
fig = px.line(
    df_jour,
    x='jour_semaine',
    y='taux_anomalie',
    text='text',
    markers=True,
    title="Évolution du taux d’anomalie selon le jour de la semaine",
    labels={'jour_semaine': 'Jour de la semaine', 'taux_anomalie': "Taux d’anomalie"}
)

fig.update_traces(
    line=dict(width=1, color='royalblue'),
    marker=dict(size=8, color='darkred'),
    textposition='top center'
)
fig.update_layout(template='plotly_white')
fig.show()


In [67]:

# Agrégation par type de boucle
boucle_anomalie = df_olt.groupby('type_boucle').agg(
    total=('is_jump_avg_dns_time', 'count'),
    anomalies=('is_jump_avg_dns_time', 'sum')
).reset_index()

boucle_anomalie['taux_anomalie'] = boucle_anomalie['anomalies'] / boucle_anomalie['total']
boucle_anomalie = boucle_anomalie.sort_values('taux_anomalie', ascending=False)

# Moyenne globale pour repère
mean_global = df_olt['is_jump_avg_dns_time'].mean()

# Graphique
fig = px.bar(
    boucle_anomalie,
    x='type_boucle',
    y='taux_anomalie',
    text=boucle_anomalie['taux_anomalie'].apply(lambda x: f"{x:.2%}"),
    title="Taux d’anomalie par type de boucle – is_jump_avg_dns_time",
    labels={'type_boucle': "Type de boucle", 'taux_anomalie': "Taux d'anomalie"}
)

fig.add_shape(
    type="line",
    x0=-0.5,
    x1=len(boucle_anomalie)-0.5,
    y0=mean_global,
    y1=mean_global,
    line=dict(color="black", dash="dash"),
)
fig.add_annotation(
    x=0,
    y=mean_global,
    text=f"Moyenne globale ({mean_global:.2%})",
    showarrow=False,
    yshift=10,
    font=dict(color="black")
)

fig.update_traces(marker_color='salmon', textposition='outside')
fig.update_layout(template="plotly_white")
fig.show()


In [68]:


# Agrégation : nombre total de lignes et anomalies par DSP
dsp_counts = df_olt.groupby('dsp').agg(
    nb_total=('is_jump_avg_dns_time', 'count'),
    nb_anomalies=('is_jump_avg_dns_time', 'sum')
).reset_index()

# Calcul du taux d’anomalie
dsp_counts['taux_anomalie'] = dsp_counts['nb_anomalies'] / dsp_counts['nb_total']

# Moyenne globale
mean_anomaly_rate = dsp_counts['taux_anomalie'].mean()

# Trié par taux décroissant
dsp_counts_sorted = dsp_counts.sort_values('taux_anomalie', ascending=False)

# Graphique
fig = px.bar(
    dsp_counts_sorted,
    x='dsp',
    y='taux_anomalie',
    title="Taux d’anomalie par DSP – is_jump_avg_dns_time",
    labels={'dsp': 'DSP', 'taux_anomalie': "Taux d'anomalie"},
    text=dsp_counts_sorted['taux_anomalie'].apply(lambda x: f"{x:.2%}")
)

# Ajout de la ligne de moyenne
fig.add_shape(
    type='line',
    x0=-0.5,
    x1=len(dsp_counts_sorted)-0.5,
    y0=mean_anomaly_rate,
    y1=mean_anomaly_rate,
    line=dict(color='black', width=2, dash='dash')
)

fig.add_annotation(
    x=0,
    y=mean_anomaly_rate,
    text=f"Moyenne globale ({mean_anomaly_rate:.2%})",
    showarrow=False,
    yshift=10
)

fig.update_traces(marker_color='salmon', textposition='outside')
fig.update_layout(template='plotly_white', xaxis_tickangle=-45)
fig.show()




is_jump_avg_latence_scoring


In [13]:


import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt

In [5]:
dataset_olt = '/content/drive/MyDrive/nexialog/data_with_anomalies_OLT_with_jump_and_DB_SCAN.parquet'
dataset_peag = "/content/drive/MyDrive/nexialog/data_with_anomalies_PEAG_with_jump_and_DB_SCAN.parquet"
df_olt = pd.read_parquet(dataset_olt)
df_peag = pd.read_parquet(dataset_peag)

In [23]:
import pandas as pd
import plotly.graph_objects as go

# Agrégation par département
df_agg = df_olt.groupby('code_departement')['is_jump_avg_latence_scoring'].sum().sort_values(ascending=False)
df_agg_pct = df_agg / df_agg.sum() * 100
df_agg_cum = df_agg_pct.cumsum()

# Seuil stratégique
seuil = 45
nb_depts_ciblés = (df_agg_cum < seuil).sum() + 1

# Création de la figure Plotly
fig = go.Figure()

# Barres pour la contribution % par département
fig.add_trace(go.Bar(
    x=df_agg_pct.index.astype(str),
    y=df_agg_pct.values,
    name="Contribution % par département",
    marker_color='rgba(55, 128, 191, 0.7)',
    hovertemplate='Département %{x}<br>Contribution: %{y:.2f}%<extra></extra>'
))

# Ligne de cumul
fig.add_trace(go.Scatter(
    x=df_agg_cum.index.astype(str),
    y=df_agg_cum.values,
    name="Cumul %",
    mode='lines+markers',
    marker=dict(color='red'),
    line=dict(shape='hv'),
    hovertemplate='Département %{x}<br>Cumul: %{y:.2f}%<extra></extra>'
))

# Lignes de seuils
fig.add_shape(
    type="line",
    x0=-0.5,
    x1=len(df_agg_pct)-0.5,
    y0=seuil,
    y1=seuil,
    line=dict(color="green", dash="dash"),
)
fig.add_shape(
    type="line",
    x0=nb_depts_ciblés-0.5,
    x1=nb_depts_ciblés-0.5,
    y0=0,
    y1=100,
    line=dict(color="gray", dash="dash"),
)

# Mises en forme
fig.update_layout(
    title="Contribution des départements aux anomalies de latence scorées",
    xaxis_title="Code département",
    yaxis_title="Pourcentage (%)",
    legend=dict(x=0.01, y=0.99),
    template="plotly_white",
    hovermode="x unified",
    height=500,
    margin=dict(l=40, r=40, t=60, b=80),
    showlegend=True,
)

fig.show()




In [22]:
# Calcul de la contribution par département
df_agg = df_olt.groupby('code_departement')['is_jump_avg_latence_scoring'].sum().sort_values(ascending=False)
df_agg_pct = df_agg / df_agg.sum() * 100
df_agg_cum = df_agg_pct.cumsum()

# Seuil de couverture
seuil = 45
nb_depts_cibles = (df_agg_cum < seuil).sum() + 1
departements_cibles = df_agg_cum.iloc[:nb_depts_cibles].index.tolist()
pourcentage_couvert = df_agg_cum.iloc[nb_depts_cibles - 1]

print(nb_depts_cibles)
print(departements_cibles)
print(pourcentage_couvert)


18
['59', '33', '62', '77', '75', '38', '13', '67', '60', '69', '31', '83', '34', '78', '45', '30', '76', '92']
45.00590414919174


In [27]:


# Regroupement
grouped = df_olt.groupby(['depts_egaux', 'meme_region'])

# Calcul des totaux et effectifs
summary = grouped['is_jump_avg_latence_scoring'].agg(
    total_anomalies='sum',
    nb_obs='count'
).reset_index()

# Taux moyen d'anomalies
summary['taux_moyen'] = summary['total_anomalies'] / summary['nb_obs'] * 100

# Heatmap Plotly
fig = px.density_heatmap(
    summary,
    x='depts_egaux',
    y='meme_region',
    z='taux_moyen',
    text_auto=True,
    color_continuous_scale='Reds',
    labels={'taux_moyen': 'Taux moyen d’anomalies (%)'},
    title="Taux moyen de sauts de latence scorés par combinaison depts_egaux × meme_region"
)
fig.update_layout(height=500)
fig.show()


In [26]:


# Lecture des données
chemin_donnees_preprocessees = "/content/drive/MyDrive/nexialog/data_preprocessed_FINAL.parquet"
donnee_brutes = pd.read_parquet(chemin_donnees_preprocessees, engine="pyarrow").sort_values(by="date_hour")

# Création de la variable model_boucle
donnee_brutes['model_boucle'] = donnee_brutes['boucle'].str[:2]

# Colonnes à analyser
colonnes = ['olt_model', 'dsp', 'pop_dns', 'depts_egaux', 'meme_region', 'model_boucle']

# Palette personnalisée
palette_switch = ['#E4001D', '#FF957C', '#0EBEBD', '#281714', '#EFDFDF']

# Dictionnaire pour stocker les répartitions
distributions_top5 = {}

for col in colonnes:
    counts = donnee_brutes[col].value_counts()
    top5 = counts.head(5)
    autres_total = counts.iloc[5:].sum()

    if autres_total > 0:
        top5['Autres'] = autres_total

    top5_percent = (top5 / counts.sum() * 100).round(2)
    distributions_top5[col] = top5_percent

    df_plot = top5_percent.reset_index()
    df_plot.columns = [col, 'Pourcentage']

    # Tracé du camembert avec Plotly
    fig = px.pie(
        df_plot,
        names=col,
        values='Pourcentage',
        title=f"Répartition de {col}",
        color_discrete_sequence=palette_switch
    )
    fig.update_traces(textinfo='percent+label', textfont_size=16, pull=[0.05]*len(df_plot))
    fig.update_layout(
        title_font_size=20,
        title_font_color='black',
        showlegend=False,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)'
    )
    fig.show()

# Optionnel : créer une table récapitulative
table_repartition_top5 = pd.concat(distributions_top5, axis=1)


In [28]:


# Regroupement par config géo + moment de la journée
grouped = df_olt.groupby(['depts_egaux', 'meme_region', 'moment_journee'])

# Calcul des anomalies et volumes
summary = grouped['is_jump_avg_latence_scoring'].agg(
    total_anomalies='sum',
    nb_obs='count'
).reset_index()

# Taux en %
summary['taux_anomalies'] = summary['total_anomalies'] / summary['nb_obs'] * 100

# Fusion des deux indicateurs géographiques pour affichage clair
summary['geo_combo'] = summary['depts_egaux'].astype(str) + '/' + summary['meme_region'].astype(str)

# Heatmap Plotly
fig = px.density_heatmap(
    summary,
    x='moment_journee',
    y='geo_combo',
    z='taux_anomalies',
    text_auto='.1f',
    color_continuous_scale='OrRd',
    labels={'taux_anomalies': 'Taux anomalies (%)'},
    title="Taux d’anomalies scorées par moment de la journée et configuration géographique"
)
fig.update_layout(height=500)
fig.show()


In [31]:
# Taux d'anomalies par DSP
dsp_summary = df_olt.groupby('dsp')['is_jump_avg_latence_scoring'].agg(['sum', 'count']).reset_index()
dsp_summary['taux_anomalies'] = dsp_summary['sum'] / dsp_summary['count'] * 100
dsp_summary = dsp_summary.sort_values(by='taux_anomalies', ascending=False)

# Taux d'anomalies par modèle d'OLT
olt_summary = df_olt.groupby('olt_model')['is_jump_avg_latence_scoring'].agg(['sum', 'count']).reset_index()
olt_summary['taux_anomalies'] = olt_summary['sum'] / olt_summary['count'] * 100
olt_summary = olt_summary.sort_values(by='taux_anomalies', ascending=False)

# Affichage clair
print("🟠 Top 10 DSP par taux d’anomalies scorées :")
display(dsp_summary.head(10))

print("\n🔵 Top 10 modèles d’OLT par taux d’anomalies scorées :")
display(olt_summary.head(10))


🟠 Top 10 DSP par taux d’anomalies scorées :


,dsp,sum,count,taux_anomalies
20,dsp_29,169,14445,1.169955
21,dsp_3,48,4480,1.071429
23,dsp_31,111,11206,0.990541
2,dsp_11,56,6281,0.891578
17,dsp_26,316,35996,0.877875
5,dsp_14,284,33816,0.839839
22,dsp_30,157,18778,0.836085
12,dsp_20,326,40703,0.800924
28,dsp_8,1656,210291,0.787480
1,dsp_10,21,2736,0.767544



🔵 Top 10 modèles d’OLT par taux d’anomalies scorées :


,olt_model,sum,count,taux_anomalies
4,M24,2052,264710,0.775188
3,M22,11018,1572629,0.700610
1,M13,4933,711384,0.693437
0,M11,4042,662695,0.609934
5,old0,1943,345558,0.562279
2,M20,129,30977,0.416438
6,old1,442,128389,0.344266


In [32]:
# Groupement
grouped = df_olt.groupby(['type_boucle', 'type_olt_model'])['is_jump_avg_latence_scoring']

# Calcul du volume d'anomalies et du taux moyen
summary = grouped.agg(
    total_anomalies='sum',
    nb_obs='count'
).reset_index()

# Ajout du taux d’anomalies
summary['taux_anomalies'] = summary['total_anomalies'] / summary['nb_obs'] * 100

# Contribution de chaque couple au total global
total_global = summary['total_anomalies'].sum()
summary['contribution_pct'] = summary['total_anomalies'] / total_global * 100

# Tri par contribution
summary = summary.sort_values(by='contribution_pct', ascending=False)

# Affichage
import pandas as pd
pd.set_option('display.max_rows', 100)
display(summary.head(15))  # Top 15 croisements les plus contributeurs


,type_boucle,type_olt_model,total_anomalies,nb_obs,taux_anomalies,contribution_pct
0,BU,New,14195,2082308,0.681696,57.799585
2,FTTH,New,7979,1160087,0.687793,32.489108
1,BU,old,2293,460108,0.498361,9.336699
3,FTTH,old,92,13839,0.664788,0.374608


In [33]:


# Agrégation par modèle OLT
olt_summary = df_olt.groupby('olt_model')['is_jump_avg_latence_scoring'].agg(['sum', 'count']).reset_index()
olt_summary['taux_anomalies'] = olt_summary['sum'] / olt_summary['count'] * 100
olt_summary = olt_summary.sort_values(by='taux_anomalies', ascending=False)

# Graphique Plotly
fig = px.bar(
    olt_summary,
    x='olt_model',
    y='taux_anomalies',
    text='taux_anomalies',
    title='Taux d’anomalies scorées par modèle d’OLT',
    labels={'olt_model': 'Modèle OLT', 'taux_anomalies': 'Taux d’anomalies (%)'},
    color='taux_anomalies',
    color_continuous_scale='OrRd'
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(
    uniformtext_minsize=8,
    uniformtext_mode='hide',
    yaxis_range=[0, olt_summary['taux_anomalies'].max() * 1.2]
)
fig.show()


is_jump_avg_score_scoring

In [35]:

# Extraire l'heure si ce n’est pas encore fait
df_olt['hour'] = df_olt['date_hour'].dt.hour

# Agrégation
hourly_anomalies = df_olt.groupby('hour')['is_jump_avg_score_scoring'].sum().sort_values(ascending=False)
hourly_pct = hourly_anomalies / hourly_anomalies.sum() * 100
hourly_cum = hourly_pct.cumsum()

# Seuil stratégique
seuil = 40
nb_heures_cibles = (hourly_cum < seuil).sum() + 1

# Graph Plotly
fig = go.Figure()

# Barres : contribution % par heure
fig.add_trace(go.Bar(
    x=hourly_pct.index.astype(str),
    y=hourly_pct.values,
    name="Contribution % par heure",
    marker_color='indianred'
))

# Courbe cumulée
fig.add_trace(go.Scatter(
    x=hourly_cum.index.astype(str),
    y=hourly_cum.values,
    name="Cumul %",
    mode='lines+markers',
    line=dict(color='black')
))

# Lignes seuils
fig.add_shape(type="line", x0=-0.5, x1=23.5, y0=seuil, y1=seuil,
              line=dict(color="green", dash="dash"))
fig.add_shape(type="line", x0=nb_heures_cibles - 0.5, x1=nb_heures_cibles - 0.5,
              y0=0, y1=100, line=dict(color="gray", dash="dash"))

# Mise en forme
fig.update_layout(
    title="Contribution des heures aux anomalies de score scoré",
    xaxis_title="Heure",
    yaxis_title="Contribution (%)",
    template="plotly_white",
    legend=dict(x=0.01, y=0.99),
    height=500
)

fig.show()



In [48]:


# Définir la condition d'anomalie (1 si anomalie, sinon 0)
df_olt['is_anomaly'] = df_olt['is_jump_avg_score_scoring'] == 1

# Calculer le nombre de clients affectés pour chaque ligne :
# Si is_anomaly est True, on prend nb_client_total, sinon 0.
df_olt['affected_clients'] = df_olt['is_anomaly'].astype(int) * df_olt['nb_client_total']

# Calcul global du nombre de clients affectés
total_clients_affected = df_olt['affected_clients'].sum()

# Nombre total de clients (global)
total_clients_all = df_olt['nb_client_total'].sum()

# Calculer le pourcentage de clients affectés en divisant le total de clients affectés par le total des clients
percentage_clients_affected = (total_clients_affected / total_clients_all) * 100

print(f"Pourcentage global de clients affectés : {percentage_clients_affected:.2f}%")


Pourcentage global de clients affectés : 0.38%


In [49]:


#  Dictionnaire de traduction anglais → français
jours_fr = {
    'Monday': 'Lundi', 'Tuesday': 'Mardi', 'Wednesday': 'Mercredi',
    'Thursday': 'Jeudi', 'Friday': 'Vendredi', 'Saturday': 'Samedi', 'Sunday': 'Dimanche'
}

#  Extraction du jour
df_olt['jour_semaine'] = df_olt['date_hour'].dt.day_name().map(jours_fr)

#  Cible
target_col = 'is_jump_avg_score_scoring'

#  Compter anomalies et volume
anomalies_jour = df_olt[df_olt[target_col] == 1]['jour_semaine'].value_counts().rename('nb_anomalies')
volume_jour = df_olt['jour_semaine'].value_counts().rename('nb_total')

#  DataFrame
df_jour = pd.concat([anomalies_jour, volume_jour], axis=1).fillna(0)
df_jour['taux_anomalie'] = df_jour['nb_anomalies'] / df_jour['nb_total']
df_jour = df_jour.reset_index().rename(columns={'index': 'jour_semaine'})

#  Ordre des jours
ordre = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche']
df_jour['jour_semaine'] = pd.Categorical(df_jour['jour_semaine'], categories=ordre, ordered=True)
df_jour = df_jour.sort_values('jour_semaine')

#  Ajout colonne texte pour annotation
df_jour['text'] = df_jour['taux_anomalie'].apply(lambda x: f"{x:.2%}")

#  Graphique
fig = px.line(
    df_jour,
    x='jour_semaine',
    y='taux_anomalie',
    text='text',
    markers=True,
    title="Évolution du taux d’anomalie selon le jour de la semaine",
    labels={'jour_semaine': 'Jour de la semaine', 'taux_anomalie': "Taux d’anomalie"}
)

fig.update_traces(
    line=dict(width=1, color='royalblue'),
    marker=dict(size=8, color='darkred'),
    textposition='top center'
)
fig.update_layout(template='plotly_white')
fig.show()


In [50]:


# Agrégation par type de boucle
boucle_anomalie = df_olt.groupby('type_boucle').agg(
    total=('is_jump_avg_score_scoring', 'count'),
    anomalies=('is_jump_avg_score_scoring', 'sum')
).reset_index()

boucle_anomalie['taux_anomalie'] = boucle_anomalie['anomalies'] / boucle_anomalie['total']
boucle_anomalie = boucle_anomalie.sort_values('taux_anomalie', ascending=False)

# Moyenne globale pour repère
mean_global = df_olt['is_jump_avg_score_scoring'].mean()

# Graphique
fig = px.bar(
    boucle_anomalie,
    x='type_boucle',
    y='taux_anomalie',
    text=boucle_anomalie['taux_anomalie'].apply(lambda x: f"{x:.2%}"),
    title="Taux d’anomalie par type de boucle – is_jump_avg_score_scoring",
    labels={'type_boucle': "Type de boucle", 'taux_anomalie': "Taux d'anomalie"}
)

fig.add_shape(
    type="line",
    x0=-0.5,
    x1=len(boucle_anomalie)-0.5,
    y0=mean_global,
    y1=mean_global,
    line=dict(color="black", dash="dash"),
)
fig.add_annotation(
    x=0,
    y=mean_global,
    text=f"Moyenne globale ({mean_global:.2%})",
    showarrow=False,
    yshift=10,
    font=dict(color="black")
)

fig.update_traces(marker_color='salmon', textposition='outside')
fig.update_layout(template="plotly_white")
fig.show()


In [51]:
i
target = "is_jump_avg_score_scoring"

summary = df_olt.groupby("week_end").agg(
    nb_observations=('week_end', 'count'),
    nb_anomalies=(target, 'sum')
).reset_index()

summary['taux_anomalies (%)'] = (summary['nb_anomalies'] / summary['nb_observations'] * 100).round(2)
summary['week_end_label'] = summary['week_end'].map({False: 'Semaine', True: 'Week-end'})

fig = px.bar(
    summary,
    x='week_end_label',
    y='taux_anomalies (%)',
    text='taux_anomalies (%)',
    labels={'week_end_label': 'Jour', 'taux_anomalies (%)': 'Taux d\'anomalie (%)'},
    title=f"Taux d'anomalies ({target}) - Semaine vs Week-end"
)
fig.update_traces(textposition='outside')
fig.update_layout(yaxis_range=[0, 100])

fig.show()

In [52]:


target = 'is_jump_avg_score_scoring'

# Données préparées
anomalies_by_model = df_olt.groupby("olt_model")[target].sum().reset_index()
anomalies_by_model.columns = ['olt_model', 'nb_anomalies']
anomalies_by_model = anomalies_by_model.sort_values(by='nb_anomalies', ascending=False)
anomalies_by_model['pct_cumul_anomalies'] = anomalies_by_model['nb_anomalies'].cumsum() / anomalies_by_model['nb_anomalies'].sum() * 100

# Graphique
fig = go.Figure()

# Barres
fig.add_trace(go.Bar(
    x=anomalies_by_model['olt_model'],
    y=anomalies_by_model['nb_anomalies'],
    name="Nombre d'anomalies",
    marker_color='rgba(55, 83, 109, 0.8)',
    text=anomalies_by_model['nb_anomalies'],
    textposition='auto'
))

# Courbe % cumulé
fig.add_trace(go.Scatter(
    x=anomalies_by_model['olt_model'],
    y=anomalies_by_model['pct_cumul_anomalies'],
    name="Pourcentage cumulé des anomalies",
    yaxis='y2',
    mode='lines+markers',
    line=dict(color='royalblue', width=2)
))

# Layout clair et responsive
fig.update_layout(
    title={
        'text': f"<b>Concentration des anomalies pour {target}</b>",
        'x': 0.5,
        'xanchor': 'center',
        'font': dict(size=20)
    },
    xaxis=dict(title="Modèle OLT", tickangle=-45),
    yaxis=dict(title="Nombre d'anomalies"),
    yaxis2=dict(title="Pourcentage cumulé des anomalies", overlaying='y', side='right', showgrid=False),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=60, r=60, t=80, b=100),
    height=500,
    template="plotly_white"
)

fig.show()

In [54]:

# Nettoyage : on enlève les éventuels NaN
df_filtered = df_olt.dropna(subset=["is_jump_avg_score_scoring", "moment_journee"]).copy()
df_filtered["is_jump_avg_score_scoring"] = df_filtered["is_jump_avg_score_scoring"].astype(int)

# Calcul du taux d'anomalie par moment de la journée
grouped = df_filtered.groupby("moment_journee").agg(
    total=('is_jump_avg_score_scoring', 'count'),
    anomalies=('is_jump_avg_score_scoring', 'sum')
).reset_index()
grouped["taux_anomalie"] = (grouped["anomalies"] / grouped["total"]) * 100

# Tri pour affichage clair
grouped = grouped.sort_values("taux_anomalie", ascending=False)

# Plot
fig = px.bar(
    grouped,
    x="moment_journee",
    y="taux_anomalie",
    text=grouped["taux_anomalie"].round(2),
    labels={"moment_journee": "Moment de la journée", "taux_anomalie": "Taux d'anomalie (%)"},
    title="Taux d'anomalies (is_jump_avg_dns_time) selon le moment de la journée"
)
fig.update_traces(textposition='outside')
fig.update_layout(yaxis_range=[0, grouped["taux_anomalie"].max() * 1.2])

fig.show()

In [58]:


# Groupement par PEAG
peag_anomalies = df_peag.groupby('peag_nro')['is_jump_avg_dns_time'].sum().sort_values(ascending=False)
peag_pct = peag_anomalies / peag_anomalies.sum() * 100
peag_cum = peag_pct.cumsum()

# Seuil stratégique (ex: 40%)
seuil = 40
nb_peag_cibles = (peag_cum < seuil).sum() + 1

# Création du graphique
fig = go.Figure()

fig.add_trace(go.Bar(
    x=peag_pct.index.astype(str),
    y=peag_pct.values,
    name="Contribution % par PEAG",
    marker_color='darkblue'
))

fig.add_trace(go.Scatter(
    x=peag_cum.index.astype(str),
    y=peag_cum.values,
    name="Cumul %",
    mode='lines+markers',
    line=dict(color='orange')
))

# Lignes seuils
fig.add_shape(type="line", x0=-0.5, x1=len(peag_pct)-0.5, y0=seuil, y1=seuil,
              line=dict(color="green", dash="dash"))
fig.add_shape(type="line", x0=nb_peag_cibles - 0.5, x1=nb_peag_cibles - 0.5,
              y0=0, y1=100, line=dict(color="gray", dash="dash"))

fig.update_layout(
    title="Contribution des PEAG aux anomalies DNS",
    xaxis_title="PEAG",
    yaxis_title="Contribution (%)",
    template="plotly_white",
    legend=dict(x=0.01, y=0.99),
    height=500
)

fig.show()


In [62]:
# Chargement du fichier déjà utilisé pour les graphes précédents
df_peag = df_peag.copy()

# Extraction de l'heure depuis la colonne 'date_hour'
df_peag['heure'] = df_peag['date_hour'].dt.hour

# Agrégation : nombre d'anomalies de DNS par heure
anomalies_par_heure = df_peag.groupby('heure')['is_anomaly_dns'].agg(['sum', 'count']).reset_index()
anomalies_par_heure['taux_anomalie'] = anomalies_par_heure['sum'] / anomalies_par_heure['count'] * 100

# Moyenne globale pour référence
moyenne_globale = anomalies_par_heure['taux_anomalie'].mean()

# Création du graphique Plotly
fig = go.Figure()

fig.add_trace(go.Bar(
    x=anomalies_par_heure['heure'],
    y=anomalies_par_heure['taux_anomalie'],
    name='Taux d’anomalies DNS (%)',
    marker_color='indianred',
    text=[f"{v:.2f}%" for v in anomalies_par_heure['taux_anomalie']],
    textposition='outside'
))

fig.add_hline(y=moyenne_globale, line_dash="dash", line_color="black",
              annotation_text=f"Moyenne globale ({moyenne_globale:.2f}%)", annotation_position="top left")

fig.update_layout(
    title="Taux d’anomalies DNS par heure de la journée",
    xaxis_title="Heure",
    yaxis_title="Taux d’anomalie (%)",
    bargap=0.2,
    template="simple_white"
)

fig.show()

In [63]:


# Conversion de la colonne date_hour en datetime
df_peag['date_hour'] = pd.to_datetime(df_peag['date_hour'], errors='coerce')

# Extraire le jour de la semaine
df_peag['jour_semaine'] = df_peag['date_hour'].dt.day_name()

# Calcul du taux d'anomalie DNS par jour de la semaine
anomalie_par_jour = df_peag.groupby('jour_semaine')['is_anomaly_dns'].mean().sort_values(ascending=False) * 100
anomalie_par_jour = anomalie_par_jour.reset_index()

# Ordre personnalisé des jours de la semaine
ordre_jours = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
anomalie_par_jour['jour_semaine'] = pd.Categorical(anomalie_par_jour['jour_semaine'], categories=ordre_jours, ordered=True)
anomalie_par_jour = anomalie_par_jour.sort_values('jour_semaine')

# Création du graphique Plotly
fig = go.Figure()
fig.add_trace(go.Bar(
    x=anomalie_par_jour['jour_semaine'],
    y=anomalie_par_jour['is_anomaly_dns'],
    text=anomalie_par_jour['is_anomaly_dns'].map(lambda x: f"{x:.2f}%"),
    textposition='outside',
    marker_color='indianred'
))

fig.update_layout(
    title="Taux d’anomalies DNS par jour de la semaine",
    xaxis_title="Jour",
    yaxis_title="Taux d'anomalie (%)",
    yaxis_tickformat=".2f",
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()


In [64]:

# Extraction de l'heure depuis la colonne 'date_hour'
df_peag['date_hour'] = pd.to_datetime(df_peag['date_hour'], errors='coerce')
df_peag['heure'] = df_peag['date_hour'].dt.hour

# Calcul du taux d'anomalies DNS par heure
anomalies_par_heure = df_peag.groupby('heure')['is_anomaly_dns'].agg(['sum', 'count']).reset_index()
anomalies_par_heure['taux_anomalie'] = anomalies_par_heure['sum'] / anomalies_par_heure['count'] * 100

# Calcul du volume total de tests DNS par heure
volume_tests_dns = df_peag.groupby('heure')['nb_test_dns'].sum().reset_index()
volume_total = volume_tests_dns['nb_test_dns'].sum()
volume_tests_dns['volume_pct'] = volume_tests_dns['nb_test_dns'] / volume_total * 100

# Fusionner les deux jeux de données
df_compare = anomalies_par_heure.merge(volume_tests_dns, on='heure')

# Création du graphique comparatif avec Plotly
import plotly.graph_objects as go

fig = go.Figure()

# Barres : taux d'anomalies
fig.add_trace(go.Bar(
    x=df_compare['heure'],
    y=df_compare['taux_anomalie'],
    name="Taux d'anomalies DNS (%)",
    marker_color='crimson',
    yaxis='y1'
))

# Ligne : % du volume de tests DNS
fig.add_trace(go.Scatter(
    x=df_compare['heure'],
    y=df_compare['volume_pct'],
    name="Volume de tests DNS (%)",
    mode='lines+markers',
    line=dict(color='royalblue'),
    yaxis='y2'
))

# Mise en forme du layout
fig.update_layout(
    title="Taux d’anomalies DNS vs Volume de tests DNS par heure",
    xaxis=dict(title="Heure"),
    yaxis=dict(title="Taux d'anomalies DNS (%)", side='left'),
    yaxis2=dict(title="% du volume de tests DNS", overlaying='y', side='right'),
    template="plotly_white",
    legend=dict(x=0.01, y=0.99)
)

fig.show()


In [8]:
# Liste des colonnes d'anomalies à analyser
colonnes = [
    "is_jump_avg_dns_time",
    "is_jump_avg_latence_scoring",
    "is_jump_avg_score_scoring",
    "is_anomaly_dns",
    "is_anomaly_latence",
    "is_anomaly_scoring"
]

# Calcul du tableau à la maille PEAG
resultats_peag = []

for col in colonnes:
    total_peag = df_olt['peag_nro'].nunique()
    df_temp = df_olt.groupby('peag_nro')[col].sum().reset_index()
    df_temp[col] = (df_temp[col] > 0).astype(int)  # 1 si au moins une anomalie sur le PEAG

    nb_anomalie = df_temp[col].sum()
    nb_sain = total_peag - nb_anomalie

    resultats_peag.append({
        "Anomalie": col,
        "PEAG sans anomalie": nb_sain,
        "PEAG avec anomalie": nb_anomalie,
        "Pourcentage avec anomalie": round(100 * nb_anomalie / total_peag, 2)
    })

# Résultat en DataFrame
import pandas as pd
df_resultats_peag = pd.DataFrame(resultats_peag)
df_resultats_peag


,Anomalie,PEAG sans anomalie,PEAG avec anomalie,Pourcentage avec anomalie
0,is_jump_avg_dns_time,221,1942,89.78
1,is_jump_avg_latence_scoring,324,1839,85.02
2,is_jump_avg_score_scoring,917,1246,57.61
3,is_anomaly_dns,16,2147,99.26
4,is_anomaly_latence,36,2127,98.34
5,is_anomaly_scoring,1064,1099,50.81


In [9]:

# Liste des colonnes d'anomalies
colonnes = [
    "is_jump_avg_dns_time",
    "is_jump_avg_latence_scoring",
    "is_jump_avg_score_scoring",
    "is_anomaly_dns",
    "is_anomaly_latence",
    "is_anomaly_scoring"
]

# Création d'un tableau résumé
res = []

for col in colonnes:
    total = len(df_peag)
    nb_1 = df_peag[col].sum()
    nb_0 = total - nb_1
    pct_1 = round(100 * nb_1 / total, 2)
    pct_0 = round(100 * nb_0 / total, 2)

    res.append({
        "Anomalie": col,
        "Valeur": 0,
        "Nombre": nb_0,
        "Pourcentage": pct_0
    })
    res.append({
        "Anomalie": col,
        "Valeur": 1,
        "Nombre": nb_1,
        "Pourcentage": pct_1
    })
    res.append({
        "Anomalie": col,
        "Valeur": "Total",
        "Nombre": total,
        "Pourcentage": 100.0
    })

# Résultat final
df_resume = pd.DataFrame(res)
display(df_resume)


,Anomalie,Valeur,Nombre,Pourcentage
0,is_jump_avg_dns_time,0,3693441,99.38
1,is_jump_avg_dns_time,1,22901,0.62
2,is_jump_avg_dns_time,Total,3716342,100.00
3,is_jump_avg_latence_scoring,0,3691783,99.34
4,is_jump_avg_latence_scoring,1,24559,0.66
5,is_jump_avg_latence_scoring,Total,3716342,100.00
6,is_jump_avg_score_scoring,0,3695448,99.44
7,is_jump_avg_score_scoring,1,20894,0.56
8,is_jump_avg_score_scoring,Total,3716342,100.00
9,is_anomaly_dns,0,3694437,99.41
